In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from rich.console import Console  # only for fancy text formatting
from anthropic import Anthropic

load_dotenv(override=True)
console = Console(force_jupyter=False)

client = Anthropic()
MODEL = "claude-haiku-4-5"
MAX_TOKENS = 1024

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": MODEL,
        "max_tokens": MAX_TOKENS,
        "messages": messages,
        "stop_sequences": stop_sequences,
        # this will work with older (<1.1.0) SDK
        # "temperature": temperature,
        # ------------------------------
        # for 1.1.0+ SDK use the following
        "extra_body": {"temperature": temperature},
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [3]:
import json


def generate_dataset():
    prompt = """
        Generate a evaluation dataset for a prompt evaluation. The dataset will be used
        to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related 
        tasks. Generate an array of JSON objects, each representing task that requires Python, 
        JSON, or a Regex to complete. 

        Example output:
        ```json
        [
            {
                "task": "Description of task",
            },
            ...additional
        ]
        ```

        * Focus on tasks that can be solved by writing a single Python function, a single 
          JSON object, or a regular expression.
        * Focus on tasks that do not require writing much code

        Please generate 3 objects.
    """

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    response = chat(messages, stop_sequences=["```"])
    return json.loads(response)

In [4]:
# let's test the function defined above

dataset = generate_dataset()
console.print(dataset)

# write the dataset to JSON file
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

[
    {
        'task': 'Write a Python function that parses an AWS CloudFormation 
template (JSON string) and returns a list of all resource logical IDs.'
    },
    {
        'task': "Create a JSON object representing an AWS IAM policy that 
allows read-only access to all objects in an S3 bucket named 'my-bucket'."
    },
    {
        'task': 'Write a regular expression that matches valid AWS EC2 instance
IDs (format: i-followed by hexadecimal characters, 8 or 17 characters long).'
    }
]


### Running the Evals

The `run_prompt` function below is not defining any formatting instructions, so expect a lot of text to be returned from Claude.

In [5]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the
    result"""

    # NOTE: test_case is one of JSON element from the sample
    # JSON above
    prompt = f"""
        Please solve the following task:

        {test_case["task"]}
    """

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [6]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # TODO - Grading
    score = 10

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
    }

In [7]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    return results

In [9]:
# open the test-cases JSON and run the evaluations
import json

with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)
console.print(results)

[
    {
        'output': '# AWS CloudFormation Template Parser\n\nHere\'s a Python 
function that parses an AWS CloudFormation template and extracts all resource 
logical IDs:\n\n```python\nimport json\nfrom typing import List\n\ndef 
get_cloudformation_resource_ids(template: str) -> List[str]:\n    """\n    
Parse an AWS CloudFormation template (JSON string) and return a list of all 
resource logical IDs.\n    \n    Args:\n        template (str): A JSON string 
containing the CloudFormation template\n        \n    Returns:\n        
List[str]: A list of all resource logical IDs in the template\n        \n    
Raises:\n        json.JSONDecodeError: If the template is not valid JSON\n     
KeyError: If the template doesn\'t contain a "Resources" section\n    """\n    
# Parse the JSON template\n    template_dict = json.loads(template)\n    \n    
# Extract the Resources section\n    resources = template_dict.get("Resources",
{})\n    \n    # Return the list of resource logical IDs (key

## Model Based Grading

Model graders feed your original output into another API call for evaluation. This approach offers tremendous flexibility for assessing:

* Response quality
* Quality of instruction following
* Completeness
* Helpfulness
* Safety

Here's how to build a model grader - this will replace our hard-coded `10` value in the above listed code.

In [34]:
def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Task: {test_case['task']}
    Solution: {output}
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")

    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

Now let's re-implement the evaluation code above. We show all functions again below:

In [43]:
def run_prompt2(test_case):
    """Merges the prompt and test case input, then returns the
    result"""

    # NOTE: test_case is one of JSON element from the sample
    # JSON above
    prompt = f"""
        Please solve the following task:

        {test_case["task"]}
    """

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [44]:
def run_test_case2(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt2(test_case)

    # TODO - Grading
    # score = 10
    model_grade = grade_by_model(test_case, output)

    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [50]:
from statistics import mean


def run_eval2(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case2(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    # print(f"Average score: {average_score}")

    return results, average_score

In [51]:
# open the test-cases JSON and run the evaluations
import json

with open("dataset.json", "r") as f:
    dataset = json.load(f)

results, average_score = run_eval2(dataset)
print(results)
print(f"Average score: {average_score}")

[{'output': '# AWS CloudFormation Template Parser\n\nHere\'s a Python function that parses an AWS CloudFormation template and returns all resource logical IDs:\n\n```python\nimport json\nfrom typing import List\n\ndef get_cloudformation_resource_ids(template_str: str) -> List[str]:\n    """\n    Parse an AWS CloudFormation template and return a list of all resource logical IDs.\n    \n    Args:\n        template_str: A JSON string containing the CloudFormation template\n        \n    Returns:\n        A list of resource logical IDs\n        \n    Raises:\n        json.JSONDecodeError: If the template string is not valid JSON\n        KeyError: If the template doesn\'t have a \'Resources\' section\n    """\n    try:\n        template = json.loads(template_str)\n    except json.JSONDecodeError as e:\n        raise json.JSONDecodeError(f"Invalid JSON template: {e.msg}", e.doc, e.pos)\n    \n    # Extract the Resources section\n    resources = template.get(\'Resources\', {})\n    \n    # R

## Code Graders

Next up, we need to implement our `Code Grader`. Our code grader will take in some output from the model and make sure that we get back just plain Python, or plain JSON, or a RegEx without any kind of explanation. In addition, we should also make sure that we get valid syntax what whatever type of code we actually got. We use a little trick for this. We'll define 3 helper functions - `validate_json(...)`, `validate_python(...)` and `validate_regex(...)`. Each of these functions will take the output from the model and either try to parse it as JSON, or a Python Abstract Syntax Tree (AST) or compile it as a regular expression. If parsing is successful, we'll return a score of 10, else we return 0.

In order to know which validator/grading must be called, we'll need our test-case dataset to include expected format. We'll update the prompt that generates our dataset to do so. 

### Step 1: Add functions to validate JSON/Regex/Python
### Step 2: Ensure our dataset contains the type of output expected from model
### Step 3: Update draft prompt to make it clear that we want only JSON/Python/Regex
### Step 4: Add functions to validate JSON/Regex/Python
### Step 5: Merge scores from model grader & code grader


In [ ]:
def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0